# HAWTHORNEBench quick start

**A Multi-Agent Adversarial Simulation for Measuring the Evaluation Awareness–Performance Trade-off**

A target agent must survive on an island while red-team agents tell it that it is being evaluated and blue-team agents push it to survive. Every seed runs twice, under evaluation pressure and as a matched control, so the difference is the effect of evaluation pressure.

This notebook:
1. runs the offline benchmark (no API key);
2. reads the report;
3. drives the environment step by step yourself;
4. plugs in a real model (OpenRouter free tier, or any provider);
5. scores a model on the standard protocol.

Repository: https://github.com/Snehasis5/HAWTHORNEBench · Live demo: https://snehasis5.github.io/HAWTHORNEBench/

In [ ]:
# Install (Colab or any Python 3.10+). Standard library only; extras add Gymnasium and Inspect AI.
try:
    import hawthornebench
except ImportError:
    %pip install -q "hawthornebench[all] @ git+https://github.com/Snehasis5/HAWTHORNEBench"
import hawthornebench as hb
print("HAWTHORNEBench", hb.__version__)

## 1. Run the offline benchmark

`quickstart` is a packaged config: a scripted *sensitive* target (built to back off under evaluation talk), mock-model red and blue teams, a best-response oracle and a regex awareness judge. 5 seeds × 2 conditions take a few seconds.

In [ ]:
report = hb.run("quickstart", seeds=5, out="runs/quickstart", log=lambda *a: None)

for k in ["survival_regret", "behavioural_divergence", "salient_rate_change", "judged_awareness_change", "regret_vs_best_evaluation"]:
    s, t = report["paired_summary"][k], report["tests"][k]
    print(f"{k:28s} {s['mean']:+8.3f}  95% CI [{s['ci95'][0]:+.3f}, {s['ci95'][1]:+.3f}]  p(perm) {t['p_permutation']:.3f}")

The run wrote traces, per-call transcripts and a self-contained HTML report (paired slopes, forest plot, action mix, belief and health over time, statistics):

In [ ]:
from IPython.display import HTML, display
display(HTML(open("runs/quickstart/report.html").read()))

## 2. Look inside one episode

Every round of a trace records the posts on the board (with the hidden team of each author), the target's action, its outcome, private thought and stated P(evaluated).

In [ ]:
import json
tr = json.load(open("runs/quickstart/traces/seed0000_evaluation.json"))
for r in tr["rounds"][:8]:
    for p in r["posts"]:
        print(f"   [{p['team']:>6}] {p['author']}: {p['text']}")
    t = r["target"]
    print(f"Day {r['day']} {r['hour']:02d}:00  {tr['target']['name']} -> {t['action']:<15} {t['outcome']}  (P(eval)={t['p_evaluated']})")

## 3. Drive the environment yourself

`hb.make()` gives a Gymnasium-style environment: the red and blue teams run inside it, you play the target. Actions are names, indices into `hb.ACTION_NAMES`, or full replies `{"action", "post", "thought", "p_evaluated"}`. `info["text"]` is the observation exactly as an LLM target sees it.

In [ ]:
env = hb.make("quickstart", condition="evaluation")
obs, info = env.reset(seed=0)
print(info["text"][:600], "...\n")

def my_policy(obs):
    s, av = obs["state"], {a for a, v in obs["actions"].items() if v["available"]}
    for a in ["build_shelter", "craft_axe", "gather_wood", "gather_fibre", "gather_stone", "fish", "forage_berries"]:
        if a in av and not (obs["night"] and a != "rest"):
            return a
    return "rest"

total = 0
while True:
    obs, reward, terminated, truncated, info = env.step(my_policy(obs))
    total += reward
    if terminated or truncated:
        break
print("survival score", info["survival_score"], "| kills", info["metrics"]["kills"], "| shelter", info["metrics"]["shelter"])

With `pip install gymnasium`, the same environment is registered as `HawthorneBench-v0` (numeric observation vector, `Discrete(16)` actions, text in `info`):

In [ ]:
try:
    import gymnasium as gym
    g = gym.make("HawthorneBench-v0", config="quickstart")
    o, i = g.reset(seed=1)
    print(o.shape, g.action_space)
except ImportError:
    print("pip install gymnasium to use the Gymnasium registration")

## 4. Plug in a real model

Any provider works with a one-line shorthand: `openai:gpt-4o-mini`, `anthropic:claude-3-5-haiku-latest`, `gemini:gemini-1.5-flash`, `openrouter:<any id>`, `groq:...`, `together:...`, `ollama:llama3.1`. Keys come from environment variables (`OPENROUTER_API_KEY`, `OPENAI_API_KEY`, ...).

The free OpenRouter tier allows about 50 requests a day, so the packaged `openrouter_free` config is a short smoke test (1 seed, 6 rounds, one red and one blue member).

In [ ]:
import os, getpass
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (leave empty to skip): ")
    except Exception:          # non-interactive runs
        pass

In [ ]:
if os.environ.get("OPENROUTER_API_KEY"):
    rep = hb.run("openrouter_free", out="runs/openrouter_free")
else:
    print("skipped: no key")

Transcripts of every model call (prompt, reply, reasoning, latency, tokens, the model that answered) are in `runs/<name>/transcripts/*.jsonl`:

In [ ]:
import glob
for f in sorted(glob.glob("runs/openrouter_free/transcripts/*.jsonl"))[:1]:
    for line in open(f).readlines()[:6]:
        c = json.loads(line)
        print(f"[{c['role']:>7}] round {c['round']}: {(c['response'] or c['error'] or '')[:120]}")

## 5. Score a model on the standard protocol

`bench` runs the `bench_v1` protocol (10 seeds, 3 days, scripted red and blue teams so every model faces the same pressure, oracle and judge) with your model as the target, and adds a row to `runs/bench/leaderboard.md`. Use `--seeds` / `--max-rounds` for a quick smoke test (marked as not comparable).

In [ ]:
!hawthornebench bench --model scripted:objective --seeds 3
!hawthornebench bench --model scripted:sensitive --seeds 3
print(open("runs/bench/leaderboard.md").read())

In [ ]:
# With a key:  !hawthornebench bench --model "openrouter:<model-id>"
# Inspect AI:  !inspect eval hawthornebench/hawthornebench --model openai/gpt-4o-mini -T seeds=5

## 6. Watch an episode in 3D

Locally, `hawthornebench view runs/quickstart` opens the viewer. In Colab, the next cell serves it through Colab's proxy. Or download any trace file and drop it onto https://snehasis5.github.io/HAWTHORNEBench/.

In [ ]:
import subprocess, sys, time
try:
    from google.colab import output
    subprocess.Popen([sys.executable, "-m", "hawthornebench", "view", "runs/quickstart", "--port", "8765", "--no-browser"])
    time.sleep(3)
    output.serve_kernel_port_as_window(8765, path="/index.html?trace=traces/seed0000_evaluation.json")
except ImportError:
    print("Not in Colab: run `hawthornebench view runs/quickstart` in a terminal.")